<a href="https://colab.research.google.com/github/JB3Ai/Colab_Sepedi_TTS/blob/main/sepedi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Consolidated Piper Training Foundation
This cell sets up the entire Python 3.10 environment, including all required ML and audio dependencies, to prevent `ModuleNotFound` and CUDA issues.

In [ ]:
print('=== 0. SYSTEM FOUNDATION: PYTHON 3.10 ===')
!sudo apt-get update -y
!sudo apt-get install -y python3.10 python3.10-dev python3.10-distutils build-essential curl espeak-ng espeak-ng-data unzip

!curl -sS https://bootstrap.pypa.io/get-pip.py -o /tmp/get-pip.py
!python3.10 /tmp/get-pip.py
!python3.10 -m pip install "pip<24.1" "setuptools<70.0" "wheel" --force-reinstall

print('\n=== 1. INSTALL AUDIO & ML STACK ===')
!python3.10 -m pip install "numpy<2.0" torch==2.1.0+cu121 torchaudio==2.1.0+cu121 --index-url https://download.pytorch.org/whl/cu121
!python3.10 -m pip install cython librosa==0.9.2 numba scipy msgpack decorator audioread pooch lazy_loader fsspec PyYAML tqdm onnxruntime piper-phonemize-fix

print('\n=== 2. LIGHTNING INSTALL ===')
!python3.10 -m pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 lightning-utilities --no-deps

print('\n=== 3. HARDWARE VERIFICATION ===')
!python3.10 -c "import torch; print(f'Torch CUDA: {torch.cuda.is_available()}'); print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"NONE\"}')"

=== 0. SYSTEM FOUNDATION: PYTHON 3.10 ===
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'python3-distutils' instead of 'pyth

In [1]:
import torch

if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.is_available()}")
    print(f"CUDA Device Name: {torch.cuda.get_device_name(0)}")
else:
    print(f"GPU is available: {torch.cuda.is_available()}. Please ensure you have a GPU runtime selected (Runtime -> Change runtime type).")

GPU is available: False. Please ensure you have a GPU runtime selected (Runtime -> Change runtime type).


In [ ]:
from pathlib import Path
import os
import shutil
import textwrap
from google.colab import drive

PIPER_BASE = Path("/content/piper")
PY_ROOT = PIPER_BASE / "src" / "python"
DATASET_SRC = Path("/content/sepedi_tts_dataset")
TRAINING_READY = Path("/content/training_ready")

print("=== 4. PIPER REPOSITORY & DATASET ===")
if not PIPER_BASE.exists():
    %cd /content
    !git clone https://github.com/rhasspy/piper.git

if not DATASET_SRC.exists():
    drive.mount("/content/drive", force_remount=True)
    !unzip -q -o /content/drive/MyDrive/sepedi_tts_dataset.zip -d /content/

TOP_ALIGN = PY_ROOT / "monotonic_align"
TOP_ALIGN.mkdir(parents=True, exist_ok=True)
(TOP_ALIGN / "__init__.py").write_text(textwrap.dedent(r'''
import torch
import numpy as np
def maximum_path(neg_cent, mask):
    device, dtype = neg_cent.device, neg_cent.dtype
    val = neg_cent.detach().cpu().numpy().astype(np.float32)
    m = mask.detach().cpu().numpy().astype(np.float32)
    path = np.zeros_like(val)
    for b in range(val.shape[0]):
        t_t = int(m[b].sum(1).clip(0,1).sum())
        t_s = int(m[b].sum(0).clip(0,1).sum())
        if t_t <= 0 or t_s <= 0: continue
        v = val[b, :t_t, :t_s]
        dp = np.full((t_t, t_s), -1e9)
        back = np.zeros((t_t, t_s), dtype=np.int32)
        dp[0, 0] = v[0, 0]
        for j in range(1, t_s): dp[0, j] = dp[0, j-1] + v[0, j]
        for i in range(1, t_t):
            for j in range(i, t_s):
                s = dp[i, j-1]
                m_ = dp[i-1, j-1]
                if m_ > s: dp[i, j] = m_ + v[i, j]; back[i, j] = 1
                else: dp[i, j] = s + v[i, j]; back[i, j] = 0
        i, j = t_t-1, t_s-1
        while j >= 0:
            path[b, i, j] = 1.0
            if i > 0 and back[i, j] == 1: i -= 1
            j -= 1
    return torch.from_numpy(path).to(device=device, dtype=dtype)
'''))

models_path = PY_ROOT / "piper_train/vits/models.py"
if models_path.exists():
    m_text = models_path.read_text()
    if "import monotonic_align" not in m_text:
        m_text = m_text.replace("from . import attentions, commons, modules, monotonic_align", "from . import attentions, commons, modules\nimport monotonic_align")
        models_path.write_text(m_text)

%cd "{PY_ROOT}"
!python3.10 -m pip install -e . --no-deps

print("\n=== 5. PREPROCESS & TRAIN ===")
if TRAINING_READY.exists(): shutil.rmtree(TRAINING_READY)
!PYTHONPATH={PY_ROOT} python3.10 -m piper_train.preprocess --language tn --dataset-format ljspeech --input-dir {DATASET_SRC} --output-dir {TRAINING_READY} --single-speaker --sample-rate 22050
!PYTHONPATH={PY_ROOT} python3.10 -m piper_train --dataset-dir {TRAINING_READY} --accelerator gpu --devices 1 --batch-size 1 --max_steps 50 --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

=== 4. PIPER REPOSITORY & DATASET ===
/content/piper/src/python
Obtaining file:///content/piper/src/python
  Preparing metadata (setup.py) ... done
  Attempting uninstall: piper_train
    Found existing installation: piper_train 1.0.0
    Uninstalling piper_train-1.0.0:
      Successfully uninstalled piper_train-1.0.0
  Running setup.py develop for piper_train

=== 5. PREPROCESS & TRAIN ===
INFO:preprocess:Single speaker dataset
INFO:preprocess:Wrote dataset config
INFO:preprocess:Processing 663 utterance(s) with 2 worker(s)
min value is  tensor(-1.0125)
max value is  tensor(1.0705)
DEBUG:piper_train:Namespace(dataset_dir='/content/training_ready', checkpoint_epochs=None, quality='medium', resume_from_single_speaker_checkpoint=None, logger=True, enable_checkpointing=True, default_root_dir='/content/drive/MyDrive/Sepedi_Voice_Output', gradient_clip_val=None, gradient_clip_algorithm=None, num_nodes=1, num_processes=None, devices='1', gpus=None, auto_select_gpus=None, tpu_cores=None, ipus

In [ ]:
import os
from google.colab import drive

print('--- 0. INSTALLING PYTHON 3.10 AND SETTING AS DEFAULT ---')
!sudo apt-get update
!sudo apt-get install -y python3.10 python3.10-dev python3.10-distutils build-essential curl

# CRITICAL FIX 1: Explicitly fetch pip for Python 3.10
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

# Set Python 3.10 as the default python3
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!sudo update-alternatives --set python3 /usr/bin/python3.10

print('\n--- 1. DOWNGRADING PIP ---')
!python3.10 -m pip install "pip<24.1"

print('\n--- 2. RECOVERING GOOGLE DRIVE & DATA ---')
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/sepedi_tts_dataset.zip -d /content/

print('\n--- 3. DOWNLOADING PIPER ---')
%cd /content
!rm -rf piper
!git clone https://github.com/rhasspy/piper.git

print('\n--- 4. APPLYING ALL INSTALLATION FIXES ---')
!apt-get update && apt-get install -y espeak-ng espeak-ng-data

%cd /content/piper/src/python

# Remove problematic requirements from setup.py and requirements.txt first
!sed -i '/piper-phonemize/d' setup.py
!sed -i '/piper-phonemize/d' requirements.txt
!sed -i '/pytorch-lightning/d' setup.py
!sed -i '/pytorch-lightning/d' requirements.txt
!sed -i '/torchmetrics/d' setup.py
!sed -i '/torchmetrics/d' requirements.txt

# Explicitly downgrade numpy before other installations to prevent conflicts
!python3.10 -m pip install numpy==1.26.4

!python3.10 -m pip install piper-phonemize-fix onnxruntime cython librosa fsspec PyYAML tqdm

# CRITICAL FIX 2: Install PyTorch with CUDA explicitly so it uses the GPU, not the CPU
!python3.10 -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install specific versions of pytorch-lightning and torchmetrics
!python3.10 -m pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 --no-deps
!python3.10 -m pip install lightning-utilities

# Install piper in editable mode (this will process its setup.py)
!python3.10 -m pip install -e . --no-deps

# Explicitly remove potentially problematic internal monotonic_align directory
# This ensures we're using the standalone monotonic-align package.
!rm -rf piper_train/vits/monotonic_align

# Install monotonic_align as a separate package from its GitHub repository
!python3.10 -m pip install git+https://github.com/rhasspy/monotonic-align.git@vits-lightning-2#egg=monotonic_align

# Modify piper_train/vits/models.py to import monotonic_align as a top-level package
# This should be done AFTER 'pip install -e .' and 'rm -rf' and 'pip install git+'
!sed -i 's/from . import attentions, commons, modules, monotonic_align/from . import attentions, commons, modules\nimport monotonic_align/' piper_train/vits/models.py

# Nuke the Python 3.12/3.10 pkg_resources bug
!sed -i 's/__import__("pkg_resources").declare_namespace(__name__)/pass/g' /usr/local/lib/python3.10/dist-packages/lightning_fabric/__init__.py

print('\n--- 5. FIXING METADATA BUG ---')
with open('/content/sepedi_tts_dataset/metadata.csv', 'r', encoding='utf-8') as f:
    lines = [line.replace('.wav|', '|') for line in f.readlines()]
with open('/content/sepedi_tts_dataset/metadata.csv', 'w', encoding='utf-8') as f:
    f.writelines(lines)

print('\n--- 6. RUNNING PREPROCESSOR ---')
# CRITICAL FIX 3: Using 'sw' (Swahili) as a proxy phonemizer to prevent eSpeak-ng crash
!python3.10 -m piper_train.preprocess \
  --language sw \
  --dataset-format ljspeech \
  --input-dir /content/sepedi_tts_dataset \
  --output-dir /content/training_ready \
  --single-speaker \
  --sample-rate 22050

print('\n--- 7. STARTING TRAINING ---')
!python3.10 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

--- 0. INSTALLING PYTHON 3.10 AND SETTING AS DEFAULT ---
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'python3-distutils' i

In [ ]:
print('--- 0. INSTALLING PYTHON 3.10 AND SETTING AS DEFAULT ---')
!sudo apt-get update
!sudo apt-get install -y python3.10 python3.10-dev python3.10-distutils build-essential

# Set Python 3.10 as the default python3
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!sudo update-alternatives --set python3 /usr/bin/python3.10

# Verify Python version
!python3 --version
!which python3

print('\n--- 1. DOWNGRADING PIP (IF NECESSARY) ---')
# Ensure pip is for python3.10 and downgraded. If python3.10-pip wasn't installed, get-pip.py is a fallback.
!python3.10 -m pip install "pip<24.1" || (echo '--- pip not found, installing with get-pip.py as fallback ---' && wget https://bootstrap.pypa.io/get-pip.py && python3.10 get-pip.py && rm get-pip.py && python3.10 -m pip install "pip<24.1")


print('\n--- 2. RECOVERING GOOGLE DRIVE & DATA ---')
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
# Re-unzip the dataset just in case Colab deleted it too (-o overwrites without asking)
!unzip -q -o /content/drive/MyDrive/sepedi_tts_dataset.zip -d /content/

print('\n--- 3. RE-DOWNLOADING PIPER ---')
%cd /content
!rm -rf piper
!git clone https://github.com/rhasspy/piper.git
!echo "--- Contents of /content/piper after cloning ---"
!ls -l /content/piper

print('\n--- 4. RE-APPLYING ALL INSTALLATION FIXES ---')
!apt-get update && apt-get install -y espeak-ng espeak-ng-data

%cd /content/piper/src/python
!echo "--- Current working directory and contents of /content/piper/src/python BEFORE modifications ---"
!pwd
!ls -l

# Remove problematic internal monotonic_align directory if it exists
!rm -rf piper_train/vits/monotonic_align
!echo "--- Contents of piper_train/vits/ after attempting to remove monotonic_align ---"
!ls -l piper_train/vits/

!sed -i '/piper-phonemize/d' setup.py
!sed -i '/piper-phonemize/d' requirements.txt
!sed -i '/pytorch-lightning/d' requirements.txt
!sed -i '/torchmetrics/d' requirements.txt

!python3.10 -m pip install piper-phonemize-fix onnxruntime cython librosa fsspec PyYAML tqdm

# Downgrade numpy to a compatible version
!python3.10 -m pip install numpy==1.26.4 --force-reinstall

!python3.10 -m pip install torch==1.13.1 torchaudio==0.13.1
!python3.10 -m pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 --no-deps
!python3.10 -m pip install lightning-utilities

# Install piper in editable mode (now it won't have its internal monotonic_align if removed successfully)
!python3.10 -m pip install -e . --no-deps

# Install monotonic_align as a separate package (the actual one we want)
!python3.10 -m pip install git+https://github.com/rhasspy/monotonic-align.git@vits-lightning-2#egg=monotonic_align

# Modify piper_train/vits/models.py to fix the import statement
# The problematic line is 'from . import attentions, commons, modules, monotonic_align'
# Replace it with two lines: original imports without monotonic_align, and a separate absolute import.
!sed -i 's/from . import attentions, commons, modules, monotonic_align/from . import attentions, commons, modules\nimport monotonic_align/' piper_train/vits/models.py
!echo "--- Content of piper_train/vits/models.py after import fix (first 15 lines) ---"
!head -n 15 piper_train/vits/models.py

# The Python 3.12 bug fix for pkg_resources is still relevant if pytorch-lightning is imported via a python3.12 environment during initial Colab boot
!sed -i 's/__import__("pkg_resources").declare_namespace(__name__)/pass/g' /usr/local/lib/python3.10/dist-packages/lightning_fabric/__init__.py

print('\n--- 5. FIXING METADATA BUG ---')
with open('/content/sepedi_tts_dataset/metadata.csv', 'r', encoding='utf-8') as f:
    lines = [line.replace('.wav|', '|') for line in f.readlines()]
with open('/content/sepedi_tts_dataset/metadata.csv', 'w', encoding='utf-8') as f:
    f.writelines(lines)

print('\n--- 6. RUNNING PREPROCESSOR ---')
!python3.10 -m piper_train.preprocess \
  --language nso \
  --dataset-format ljspeech \
  --input-dir /content/sepedi_tts_dataset \
  --output-dir /content/training_ready \
  --single-speaker \
  --sample-rate 22050

print('\n--- 7. STARTING TRAINING ---')
!python3.10 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

Streaming output truncated to the last 5000 lines.
Reading state information... Done
The following additional packages will be installed:
  libespeak-ng1 libpcaudio0 libsonic0
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 43 not upgraded.
Need to get 4,526 kB of archives.
After this operation, 11.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpcaudio0 amd64 1.1-6build2 [8,956 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsonic0 amd64 0.2.0-11build1 [10.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 espeak-ng-data amd64 1.50+dfsg-10ubuntu0.1 [3,956 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libespeak-ng1 amd64 1.50+dfsg-10ubuntu0.1 [207 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 espeak-ng amd64 1.50+dfsg-10ubuntu0.1 [343 kB]
Fetched

In [ ]:
print('--- 0. INSTALLING PYTHON 3.10 AND SETTING AS DEFAULT ---')
!sudo apt-get update
!sudo apt-get install -y python3.10 python3.10-dev python3.10-distutils build-essential

# Set Python 3.10 as the default python3
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!sudo update-alternatives --set python3 /usr/bin/python3.10

# Verify Python version
!python3 --version
!which python3

print('\n--- 1. DOWNGRADING PIP (IF NECESSARY) ---')
# Ensure pip is for python3.10 and downgraded
!python3.10 -m pip install "pip<24.1"

print('\n--- 2. RECOVERING GOOGLE DRIVE & DATA ---')
import os
from google.colab import drive

drive.mount('/content/drive')
# Re-unzip the dataset just in case Colab deleted it too (-o overwrites without asking)
!unzip -q -o /content/drive/MyDrive/sepedi_tts_dataset.zip -d /content/

print('\n--- 3. RE-DOWNLOADING PIPER ---')
%cd /content
!rm -rf piper
!git clone https://github.com/rhasspy/piper.git

print('\n--- 4. RE-APPLYING ALL INSTALLATION FIXES ---')
!apt-get update && apt-get install -y espeak-ng espeak-ng-data

%cd /content/piper/src/python
!sed -i '/piper-phonemize/d' setup.py
!sed -i '/piper-phonemize/d' requirements.txt
!sed -i '/pytorch-lightning/d' requirements.txt
!sed -i '/torchmetrics/d' requirements.txt

!python3.10 -m pip install piper-phonemize-fix onnxruntime cython librosa numpy fsspec PyYAML tqdm
!python3.10 -m pip install torch==1.13.1 torchaudio==0.13.1
!python3.10 -m pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 --no-deps
!python3.10 -m pip install lightning-utilities
!python3.10 -m pip install -e . --no-deps

# Fix monotonic_align import path issue due to double-nesting after installation
!sed -i 's/from .monotonic_align.core import maximum_path_c/from .core import maximum_path_c/g' piper_train/vits/monotonic_align/__init__.py

# Build the alignment tool using pip install . for better dependency resolution
!cd piper_train/vits/monotonic_align && python3.10 -m pip install .

# The Python 3.12 bug fix for pkg_resources is still relevant if pytorch-lightning is imported via a python3.12 environment during initial Colab boot
!sed -i 's/__import__("pkg_resources").declare_namespace(__name__)/pass/g' /usr/local/lib/python3.10/dist-packages/lightning_fabric/__init__.py

print('\n--- 5. FIXING METADATA BUG ---')
with open('/content/sepedi_tts_dataset/metadata.csv', 'r', encoding='utf-8') as f:
    lines = [line.replace('.wav|', '|') for line in f.readlines()]
with open('/content/sepedi_tts_dataset/metadata.csv', 'w', encoding='utf-8') as f:
    f.writelines(lines)

print('\n--- 6. RUNNING PREPROCESSOR ---')
!python3.10 -m piper_train.preprocess \
  --language nso \
  --dataset-format ljspeech \
  --input-dir /content/sepedi_tts_dataset \
  --output-dir /content/training_ready \
  --single-speaker \
  --sample-rate 22050

print('\n--- 7. STARTING TRAINING ---')
!python3.10 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

--- 0. INSTALLING PYTHON 3.10 AND SETTING AS DEFAULT ---
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'python3-distutils' i

In [ ]:

print('--- Installing pip for python3.10 using get-pip.py ---')
!wget https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py
!rm get-pip.py

print('\n--- Verifying pip for python3.10 ---')
!python3.10 -m pip --version

print('\n--- Please manually re-run the main setup cell (a4a4a676) now ---')

In [ ]:
# Clone the Piper repository
!git clone [https://github.com/rhasspy/piper.git](https://github.com/rhasspy/piper.git)
%cd piper/src/python

# Install Linux phoneme dependencies
!apt-get update && apt-get install -y espeak-ng

# CRITICAL FIX 1: Downgrade pip to bypass strict metadata checks
!python3 -m pip install "pip<24.1"

# CRITICAL FIX 2: Install the modern community-fixed phonemizer
!sed -i '/piper-phonemize/d' setup.py
!sed -i '/piper-phonemize/d' requirements.txt
!pip install piper-phonemize-fix

# CRITICAL FIX 3: Manually install dependencies to completely bypass Piper's broken dependency tree
!pip install cython librosa numpy onnxruntime
!pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 --no-deps
!pip install -e . --no-deps

# CRITICAL FIX 4: Nuke the Python 3.12 pkg_resources bug directly inside PyTorch Lightning Fabric
!sed -i 's/__import__("pkg_resources").declare_namespace(__name__)/pass/g' /usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py

# Build the alignment tool (with the VITS directory fix)
!cd piper_train/vits/monotonic_align && mkdir -p monotonic_align && python setup.py build_ext --inplace
print("Piper AI installed successfully!")

/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `git clone [https://github.com/rhasspy/piper.git](https://github.com/rhasspy/piper.git)'
[Errno 2] No such file or directory: 'piper/src/python'
/content
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,026 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,221 kB]
Get:11 http://s

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 44.6 MB/s eta 0:00:00


KeyboardInterrupt: 

In [ ]:
%cd /content/piper/src/python

print("--- 1. RUNNING PREPROCESSOR ---")
!python -m piper_train.preprocess \
  --language st \
  --dataset-format ljspeech \
  --input-dir /content/sepedi_tts_dataset \
  --output-dir /content/training_ready \
  --single-speaker \
  --sample-rate 22050

print("--- 2. STARTING TRAINING ---")
!python -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

In [ ]:
import os
from google.colab import drive

print("--- 0. RECOVERING GOOGLE DRIVE & DATA ---")
drive.mount('/content/drive')
# Re-unzip the dataset just in case Colab deleted it too (-o overwrites without asking)
!unzip -q -o /content/drive/MyDrive/sepedi_tts_dataset.zip -d /content/

print("--- 1. RE-DOWNLOADING PIPER ---")
%cd /content
!rm -rf piper
!git clone https://github.com/rhasspy/piper.git

print("--- 2. RE-APPLYING ALL INSTALLATION FIXES ---")
!apt-get update && apt-get install -y espeak-ng
!python3 -m pip install "pip<24.1"

%cd /content/piper/src/python
!sed -i '/piper-phonemize/d' setup.py
!sed -i '/piper-phonemize/d' requirements.txt
!sed -i '/pytorch-lightning/d' requirements.txt
!sed -i '/torchmetrics/d' requirements.txt

!pip install piper-phonemize-fix onnxruntime cython librosa numpy
!pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 --no-deps
!pip install -e . --no-deps

# Nuke the Python 3.12 bug
!sed -i 's/__import__("pkg_resources").declare_namespace(__name__)/pass/g' /usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py

# Build the alignment tool
!cd piper_train/vits/monotonic_align && mkdir -p monotonic_align && python setup.py build_ext --inplace

print("--- 3. FIXING METADATA BUG ---")
with open('/content/sepedi_tts_dataset/metadata.csv', 'r', encoding='utf-8') as f:
    lines = [line.replace('.wav|', '|') for line in f.readlines()]
with open('/content/sepedi_tts_dataset/metadata.csv', 'w', encoding='utf-8') as f:
    f.writelines(lines)

print("--- 4. RUNNING PREPROCESSOR ---")
!python -m piper_train.preprocess \
  --language st \
  --dataset-format ljspeech \
  --input-dir /content/sepedi_tts_dataset \
  --output-dir /content/training_ready \
  --single-speaker \
  --sample-rate 22050

print("--- 5. STARTING TRAINING ---")
!python -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

In [ ]:
%cd /content/piper/src/python

print("--- 1. INSTALLING MISSING HELPER ---")
!pip install lightning-utilities typing-extensions

print("--- 2. RUNNING PREPROCESSOR (WITH SWAHILI PHONETICS) ---")
!python -m piper_train.preprocess \
  --language sw \
  --dataset-format ljspeech \
  --input-dir /content/sepedi_tts_dataset \
  --output-dir /content/training_ready \
  --single-speaker \
  --sample-rate 22050

print("--- 3. STARTING TRAINING ---")
!python -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

/content/piper/src/python
--- 1. INSTALLING MISSING HELPER ---
--- 2. RUNNING PREPROCESSOR (WITH SWAHILI PHONETICS) ---
INFO:preprocess:Single speaker dataset
INFO:preprocess:Wrote dataset config
INFO:preprocess:Processing 663 utterance(s) with 2 worker(s)

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/content/piper/src/python/piper_train/preprocess.py", l

In [ ]:
%cd /content/piper/src/python

print("--- 1. PATCHING TORCHMETRICS ---")
!sed -i '1i import pkgutil; import zipimport; getattr(pkgutil, "ImpImporter", None) or setattr(pkgutil, "ImpImporter", zipimport.zipimporter)' /usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/imports.py

print("--- 2. STARTING TRAINING ---")
!python -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

In [ ]:
/content/piper/src/python
--- 1. PATCHING TORCHMETRICS ---
--- 2. STARTING TRAINING ---
Traceback (most recent call last):
  File "/usr/lib/python3/dist-packages/pkg_resources/__init__.py", line 2208, in _handle_ns
    loader = importer.find_spec(packageName).loader
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'loader'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/piper/src/python/piper_train/__main__.py", line 7, in <module>
    from pytorch_lightning import Trainer
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/__init__.py", line 35, in <module>
    from pytorch_lightning.callbacks import Callback  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/__init__.py", line 14, in <module>
    from pytorch_lightning.callbacks.batch_size_finder import BatchSizeFinder
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/batch_size_finder.py", line 24, in <module>
    from pytorch_lightning.callbacks.callback import Callback
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/callback.py", line 25, in <module>
    from pytorch_lightning.utilities.types import STEP_OUTPUT
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/__init__.py", line 23, in <module>
    from pytorch_lightning.utilities.imports import (  # noqa: F401
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/imports.py", line 28, in <module>
    _TORCHMETRICS_GREATER_EQUAL_0_11 = compare_version("torchmetrics", operator.ge, "0.11.0")  # using new API with task
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning_utilities/core/imports.py", line 78, in compare_version
    pkg = importlib.import_module(package)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/__init__.py", line 14, in <module>
    from torchmetrics import functional  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/functional/__init__.py", line 14, in <module>
    from torchmetrics.functional.audio.pit import permutation_invariant_training, pit_permutate
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/functional/audio/__init__.py", line 14, in <module>
    from torchmetrics.functional.audio.pit import permutation_invariant_training, pit_permutate  # noqa: F401
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/functional/audio/pit.py", line 22, in <module>
    from torchmetrics.utilities.imports import _SCIPY_AVAILABLE
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/__init__.py", line 1, in <module>
    from torchmetrics.utilities.checks import check_forward_full_state_property  # noqa: F401
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/checks.py", line 25, in <module>
    from torchmetrics.utilities.data import select_topk, to_onehot
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/data.py", line 19, in <module>
    from torchmetrics.utilities.imports import _TORCH_GREATER_EQUAL_1_12, _XLA_AVAILABLE
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/imports.py", line 23, in <module>
    from pkg_resources import DistributionNotFound, get_distribution
  File "/usr/lib/python3/dist-packages/pkg_resources/__init__.py", line 3266, in <module>
    @_call_aside
     ^^^^^^^^^^^
  File "/usr/lib/python3/dist-packages/pkg_resources/__init__.py", line 3241, in _call_aside
    f(*args, **kwargs)
  File "/usr/lib/python3/dist-packages/pkg_resources/__init__.py", line 3292, in _initialize_master_working_set
    tuple(
  File "/usr/lib/python3/dist-packages/pkg_resources/__init__.py", line 3293, in <genexpr>
    dist.activate(replace=False)
  File "/usr/lib/python3/dist-packages/pkg_resources/__init__.py", line 2798, in activate
    declare_namespace(pkg)
  File "/usr/lib/python3/dist-packages/pkg_resources/__init__.py", line 2296, in declare_namespace
    _handle_ns(packageName, path_item)
  File "/usr/lib/python3/dist-packages/pkg_resources/__init__.py", line 2213, in _handle_ns
    loader = importer.find_module(packageName)
             ^^^^^^^^^^^^^^^^^^^^
AttributeError: 'FileFinder' object has no attribute 'find_module'

### Attempting to fix `pkg_resources` error by downgrading Python to 3.10

In [ ]:
print('--- 0. INSTALLING PYTHON 3.10 ---')
!sudo apt-get update
!sudo apt-get install -y python3.10 python3.10-dev python3.10-distutils

# Update alternatives to use python3.10
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1

# Verify Python version
!python3 --version

print('\n--- 1. DOWNGRADING PIP ---')
!python3 -m pip install "pip<24.1"

print('\n--- 2. RECOVERING GOOGLE DRIVE & DATA ---')
import os
from google.colab import drive

drive.mount('/content/drive')
# Re-unzip the dataset just in case Colab deleted it too (-o overwrites without asking)
!unzip -q -o /content/drive/MyDrive/sepedi_tts_dataset.zip -d /content/

print('\n--- 3. RE-DOWNLOADING PIPER ---')
%cd /content
!rm -rf piper
!git clone https://github.com/rhasspy/piper.git

print('\n--- 4. RE-APPLYING ALL INSTALLATION FIXES ---')
!apt-get update && apt-get install -y espeak-ng

%cd /content/piper/src/python
!sed -i '/piper-phonemize/d' setup.py
!sed -i '/piper-phonemize/d' requirements.txt
!sed -i '/pytorch-lightning/d' requirements.txt
!sed -i '/torchmetrics/d' requirements.txt

!python3 -m pip install piper-phonemize-fix onnxruntime cython librosa numpy
!python3 -m pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 --no-deps
!python3 -m pip install -e . --no-deps

# Nuke the Python 3.12 bug (this path may differ slightly for 3.10, but it's a good starting point)
!sed -i 's/__import__("pkg_resources").declare_namespace(__name__)/pass/g' /usr/local/lib/python3.10/dist-packages/lightning_fabric/__init__.py

# Build the alignment tool
!cd piper_train/vits/monotonic_align && mkdir -p monotonic_align && python3 setup.py build_ext --inplace

print('\n--- 5. FIXING METADATA BUG ---')
with open('/content/sepedi_tts_dataset/metadata.csv', 'r', encoding='utf-8') as f:
    lines = [line.replace('.wav|', '|') for line in f.readlines()]
with open('/content/sepedi_tts_dataset/metadata.csv', 'w', encoding='utf-8') as f:
    f.writelines(lines)

print('\n--- 6. RUNNING PREPROCESSOR ---')
!python3 -m piper_train.preprocess \
  --language st \
  --dataset-format ljspeech \
  --input-dir /content/sepedi_tts_dataset \
  --output-dir /content/training_ready \
  --single-speaker \
  --sample-rate 22050

print('\n--- 7. STARTING TRAINING ---')
!python3 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

### Retrying Python 3.10 Downgrade and Training

In [ ]:
print('--- 0. INSTALLING PYTHON 3.10 AND SETTING AS DEFAULT ---')
!sudo apt-get update
!sudo apt-get install -y python3.10 python3.10-dev python3.10-distutils

# Set Python 3.10 as the default python3
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!sudo update-alternatives --set python3 /usr/bin/python3.10

# Verify Python version
!python3 --version
!which python3

print('\n--- 1. DOWNGRADING PIP ---')
# Ensure pip is for python3.10
!python3 -m pip install "pip<24.1"

print('\n--- 2. RECOVERING GOOGLE DRIVE & DATA ---')
import os
from google.colab import drive

drive.mount('/content/drive')
# Re-unzip the dataset just in case Colab deleted it too (-o overwrites without asking)
!unzip -q -o /content/drive/MyDrive/sepedi_tts_dataset.zip -d /content/

print('\n--- 3. RE-DOWNLOADING PIPER ---')
%cd /content
!rm -rf piper
!git clone https://github.com/rhasspy/piper.git

print('\n--- 4. RE-APPLYING ALL INSTALLATION FIXES ---')
!apt-get update && apt-get install -y espeak-ng

%cd /content/piper/src/python
!sed -i '/piper-phonemize/d' setup.py
!sed -i '/piper-phonemize/d' requirements.txt
!sed -i '/pytorch-lightning/d' requirements.txt
!sed -i '/torchmetrics/d' requirements.txt

!python3 -m pip install piper-phonemize-fix onnxruntime cython librosa numpy
!python3 -m pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 --no-deps
!python3 -m pip install -e . --no-deps

# The Python 3.12 bug fix is now less relevant if we are on Python 3.10, but I'll keep the sed commands
# targeting common library paths. If issues persist, this step might need adjustment or removal.
# Nuke the Python 3.12 bug (this path may differ slightly for 3.10)
!sed -i 's/__import__("pkg_resources").declare_namespace(__name__)/pass/g' /usr/local/lib/python3.10/dist-packages/lightning_fabric/__init__.py

# Build the alignment tool
!cd piper_train/vits/monotonic_align && mkdir -p monotonic_align && python3 setup.py build_ext --inplace

print('\n--- 5. FIXING METADATA BUG ---')
with open('/content/sepedi_tts_dataset/metadata.csv', 'r', encoding='utf-8') as f:
    lines = [line.replace('.wav|', '|') for line in f.readlines()]
with open('/content/sepedi_tts_dataset/metadata.csv', 'w', encoding='utf-8') as f:
    f.writelines(lines)

print('\n--- 6. RUNNING PREPROCESSOR ---')
!python3 -m piper_train.preprocess \
  --language st \
  --dataset-format ljspeech \
  --input-dir /content/sepedi_tts_dataset \
  --output-dir /content/training_ready \
  --single-speaker \
  --sample-rate 22050

print('\n--- 7. STARTING TRAINING ---')
!python3 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

### Final attempt: Ensuring Python 3.10 and its `pip` are correctly configured

In [ ]:
print('--- 0. INSTALLING PYTHON 3.10 AND SETTING AS DEFAULT ---')
!sudo apt-get update
!sudo apt-get install -y python3.10 python3.10-dev python3.10-distutils python3.10-pip

# Set Python 3.10 as the default python3
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!sudo update-alternatives --set python3 /usr/bin/python3.10

# Verify Python version
!python3 --version
!which python3

print('\n--- 1. DOWNGRADING PIP (IF NECESSARY) ---')
# Ensure pip is for python3.10 and downgraded
!python3.10 -m pip install "pip<24.1"

print('\n--- 2. RECOVERING GOOGLE DRIVE & DATA ---')
import os
from google.colab import drive

drive.mount('/content/drive')
# Re-unzip the dataset just in case Colab deleted it too (-o overwrites without asking)
!unzip -q -o /content/drive/MyDrive/sepedi_tts_dataset.zip -d /content/

print('\n--- 3. RE-DOWNLOADING PIPER ---')
%cd /content
!rm -rf piper
!git clone https://github.com/rhasspy/piper.git

print('\n--- 4. RE-APPLYING ALL INSTALLATION FIXES ---')
!apt-get update && apt-get install -y espeak-ng espeak-ng-data

%cd /content/piper/src/python
!sed -i '/piper-phonemize/d' setup.py
!sed -i '/piper-phonemize/d' requirements.txt
!sed -i '/pytorch-lightning/d' requirements.txt
!sed -i '/torchmetrics/d' requirements.txt

!python3.10 -m pip install piper-phonemize-fix onnxruntime cython librosa numpy
!python3.10 -m pip install torch==1.13.1 torchaudio==0.13.1
!python3.10 -m pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 --no-deps
!python3.10 -m pip install lightning-utilities
!python3.10 -m pip install -e . --no-deps

# Build the alignment tool using pip install . for better dependency resolution
!cd piper_train/vits/monotonic_align && python3.10 -m pip install .

# The Python 3.12 bug fix for pkg_resources is still relevant if pytorch-lightning is imported via a python3.12 environment during initial Colab boot
!sed -i 's/__import__("pkg_resources").declare_namespace(__name__)/pass/g' /usr/local/lib/python3.10/dist-packages/lightning_fabric/__init__.py

print('\n--- 5. FIXING METADATA BUG ---')
with open('/content/sepedi_tts_dataset/metadata.csv', 'r', encoding='utf-8') as f:
    lines = [line.replace('.wav|', '|') for line in f.readlines()]
with open('/content/sepedi_tts_dataset/metadata.csv', 'w', encoding='utf-8') as f:
    f.writelines(lines)

print('\n--- 6. RUNNING PREPROCESSOR ---')
!python3.10 -m piper_train.preprocess \
  --language st \
  --dataset-format ljspeech \
  --input-dir /content/sepedi_tts_dataset \
  --output-dir /content/training_ready \
  --single-speaker \
  --sample-rate 22050

print('\n--- 7. STARTING TRAINING ---')
!python3.10 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

In [ ]:
print('--- Installing pip for python3.10 using get-pip.py ---')
!wget https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py
!rm get-pip.py

print('\n--- Verifying pip for python3.10 ---')
!python3.10 -m pip --version

# Now re-run the main setup cell (bd776f7c) to continue with all installations
print('\n--- Re-running main setup cell (bd776f7c) ---')
get_ipython().run_cell_with_id('bd776f7c')

In [ ]:
print('Checking pip version for python3.10:')
!python3.10 -m pip --version

In [ ]:
print('--- Installing pip for python3.10 ---')
!python3.10 -m ensurepip --default-pip

print('\n--- Verifying pip for python3.10 ---')
!python3.10 -m pip --version

# Now re-run the main setup cell (bd776f7c) to continue
print('\n--- Re-running main setup cell (bd776f7c) ---')
get_ipython().run_cell(cell_id='bd776f7c')

In [ ]:
import os
from google.colab import drive

print('--- 0. SETTING UP PYTHON 3.10 ---')
!sudo apt-get update
!sudo apt-get install -y python3.10 python3.10-dev python3.10-distutils build-essential curl
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!sudo update-alternatives --set python3 /usr/bin/python3.10

print('\n--- 1. INSTALLING PIP & CORE LIBRARIES ---')
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10
!python3.10 -m pip install "pip<24.1" "numpy<2.0" cython==0.29.37 scipy==1.10.1

print('\n--- 2. MOUNTING DRIVE & DATA ---')
drive.mount('/content/drive', force_remount=True)
!unzip -q -o /content/drive/MyDrive/sepedi_tts_dataset.zip -d /content/

print('\n--- 3. DOWNLOADING PIPER ---')
%cd /content
!rm -rf piper
!git clone https://github.com/rhasspy/piper.git

print('\n--- 4. INSTALLING PIPER TRAIN DEPENDENCIES ---')
!apt-get update && apt-get install -y espeak-ng espeak-ng-data
%cd /content/piper/src/python

# Clean broken dependencies from setup
!sed -i '/piper-phonemize/d' setup.py
!sed -i '/piper-phonemize/d' requirements.txt
!python3.10 -m pip install piper-phonemize-fix onnxruntime librosa fsspec PyYAML tqdm
!python3.10 -m pip install torch==1.13.1 torchaudio==0.13.1 --index-url https://download.pytorch.org/whl/cu117
!python3.10 -m pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 lightning-utilities --no-deps

print('\n--- 5. INSTALLING MONOTONIC ALIGNMENT (STANDALONE) ---')
# Install as a global package from the lightning-compatible branch
!python3.10 -m pip install git+https://github.com/rhasspy/monotonic-align.git@vits-lightning-2

# Install piper_train in editable mode
!python3.10 -m pip install -e .

# CRITICAL: Remove the internal folder so Python uses the globally installed package
!rm -rf piper_train/vits/monotonic_align

# Patch models.py to use global import instead of relative
!sed -i 's/from . import attentions, commons, modules, monotonic_align/from . import attentions, commons, modules\nimport monotonic_align/g' piper_train/vits/models.py

# Fix namespace bug for Python 3.10
!sed -i 's/__import__("pkg_resources").declare_namespace(__name__)/pass/g' /usr/local/lib/python3.10/dist-packages/lightning_fabric/__init__.py 2>/dev/null || true

print('\n--- 6. RUNNING PREPROCESSOR (SETSWANA PROXY) ---')
# 'tn' (Setswana) is natively supported and linguistically a very close relative to Sepedi
!rm -rf /content/training_ready
!python3.10 -m piper_train.preprocess \
  --language tn \
  --dataset-format ljspeech \
  --input-dir /content/sepedi_tts_dataset \
  --output-dir /content/training_ready \
  --single-speaker \
  --sample-rate 22050

print('\n--- 7. STARTING TRAINING ---')
!python3.10 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

--- 0. SETTING UP PYTHON 3.10 ---
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'python3-distutils' instead of 'python3.10-d

In [ ]:
%cd /content/piper/src/python

print('--- 1. RUNNING PREPROCESSOR (USING SPANISH PHONETICS) ---')
# Clear out the half-finished folder from the last attempt
!rm -rf /content/training_ready

!python3.10 -m piper_train.preprocess \
  --language es \
  --dataset-format ljspeech \
  --input-dir /content/sepedi_tts_dataset \
  --output-dir /content/training_ready \
  --single-speaker \
  --sample-rate 22050

print('\n--- 2. STARTING TRAINING ---')
!python3.10 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

[Errno 2] No such file or directory: '/content/piper/src/python'
/content
--- 1. RUNNING PREPROCESSOR (USING SPANISH PHONETICS) ---
/usr/bin/python3.10: Error while finding module specification for 'piper_train.preprocess' (ModuleNotFoundError: No module named 'piper_train')

--- 2. STARTING TRAINING ---
/usr/bin/python3.10: No module named piper_train


In [ ]:
%cd /content/piper/src/python

print('--- 1. FIXING MONOTONIC ALIGNMENT ---')
# Build it manually and move the compiled .so file to the correct directory
!cd piper_train/vits/monotonic_align && mkdir -p piper_train/vits/monotonic_align && python3.10 setup.py build_ext --inplace
!cd piper_train/vits/monotonic_align && mv piper_train/vits/monotonic_align/core*.so .

print('\n--- 2. RUNNING PREPROCESSOR (WITH SPANISH PHONETICS) ---')
!python3.10 -m piper_train.preprocess \
  --language es \
  --dataset-format ljspeech \
  --input-dir /content/sepedi_tts_dataset \
  --output-dir /content/training_ready \
  --single-speaker \
  --sample-rate 22050

print('\n--- 3. STARTING TRAINING ---')
!python3.10 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

/content/piper/src/python
--- 1. FIXING MONOTONIC ALIGNMENT ---
/bin/bash: line 1: cd: piper_train/vits/monotonic_align: No such file or directory
/bin/bash: line 1: cd: piper_train/vits/monotonic_align: No such file or directory

--- 2. RUNNING PREPROCESSOR (WITH SPANISH PHONETICS) ---
INFO:preprocess:Single speaker dataset
INFO:preprocess:Wrote dataset config
INFO:preprocess:Processing 663 utterance(s) with 2 worker(s)

--- 3. STARTING TRAINING ---
Traceback (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/content/piper/src/python/piper_train/__main__.py", line 10, in <module>
    from .vits.lightning import VitsModel
  File "/content/piper/src/python/piper_train/vits/lightning.py", line 15, in <module>
    from .models import MultiPeriodDiscriminator, SynthesizerTrn
  File "/content/piper/src/

In [ ]:
print('--- Listing installed eSpeak-ng voices ---')
!espeak-ng --voices

--- Listing installed eSpeak-ng voices ---
Pty Language       Age/Gender VoiceName          File                 Other Languages
 5  af              --/M      Afrikaans          gmw/af               
 5  am              --/M      Amharic            sem/am               
 5  an              --/M      Aragonese          roa/an               
 5  ar              --/M      Arabic             sem/ar               
 5  as              --/M      Assamese           inc/as               
 5  az              --/M      Azerbaijani        trk/az               
 5  ba              --/M      Bashkir            trk/ba               
 5  bg              --/M      Bulgarian          zls/bg               
 5  bn              --/M      Bengali            inc/bn               
 5  bpy             --/M      Bishnupriya_Manipuri inc/bpy              
 5  bs              --/M      Bosnian            zls/bs               
 5  ca              --/M      Catalan            roa/ca               
 5  cmn          

In [ ]:
import numpy as np
print(f"NumPy version: {np.__version__}")

NumPy version: 2.0.2


In [ ]:
print('--- Installing pip for python3.10 using get-pip.py ---')
!wget https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py
!rm get-pip.py

print('\n--- Verifying pip for python3.10 ---')
!python3.10 -m pip --version

print('\n--- Please manually re-run the main setup cell (a4a4a676) now ---')

--- Installing pip for python3.10 using get-pip.py ---
--2026-06-22 00:47:55--  https://bootstrap.pypa.io/get-pip.py
Resolving bootstrap.pypa.io (bootstrap.pypa.io)... 151.101.0.175, 151.101.64.175, 151.101.128.175, ...
Connecting to bootstrap.pypa.io (bootstrap.pypa.io)|151.101.0.175|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2226848 (2.1M) [text/x-python]
Saving to: ‘get-pip.py’

get-pip.py          100%[===================>]   2.12M  --.-KB/s    in 0.07s   

2026-06-22 00:47:55 (29.6 MB/s) - ‘get-pip.py’ saved [2226848/2226848]

  Using cached pip-26.1.2-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0

--- Verifying pip for python3.10 ---
pip 26.1.2 from /usr/local/lib/python3.10/dist-packages/pip (python 3.10)

--- Please manually re-run the main setup cell (a4a4a676) now ---


In [ ]:
print('--- Installing joblib ---')
!python3.10 -m pip install joblib

--- Installing joblib ---


In [ ]:
!python3.10 -m pip install fsspec PyYAML tqdm

In [ ]:
%cd /content/piper
!make

/content/piper
cmake -Bbuild -DCMAKE_INSTALL_PREFIX=install
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMake Warning (dev) at /usr/local/lib/python3.12/dist-packages/cmake/data/share/cmake-3.31/Modules/ExternalProject/shared_internal_commands.cmake:1276 (message):
  The DOWNLOAD_EXTRACT_TIMESTAMP option was not given and policy CMP0135 is
  not set.  The policy's OLD behavior will be used.  When using a URL
  download, the timestamps of extracted files should preferably be that of
  the time of extraction, otherwise code that depend

In [ ]:
%cd /content/piper/build
!ctest

[Errno 2] No such file or directory: '/content/piper/build'
/content
*********************************
No test configuration file found!
*********************************
Usage

  ctest [options]



In [ ]:
import os

%cd /content

# Ensure piper directory exists, re-download if necessary
if not os.path.exists('piper'):
    print('--- Re-downloading Piper as directory not found ---')
    !rm -rf piper
    !wget -O /content/piper.zip https://github.com/rhasspy/piper/archive/refs/heads/master.zip
    !unzip -q /content/piper.zip -d /content/
    !mv /content/piper-master /content/piper
else:
    print('--- Piper directory already exists ---')

%cd /content/piper

# Clean up previous build to ensure a fresh start
!rm -rf build

# Run cmake commands to build the project and create the build directory
!cmake -Bbuild -DCMAKE_INSTALL_PREFIX=install
!cmake --build build --config Release

# Change to the build directory and run ctest without the --config argument
%cd build
!ctest

/content
--- Re-downloading Piper as directory not found ---
--2026-06-22 06:26:22--  https://github.com/rhasspy/piper/archive/refs/heads/master.zip
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/rhasspy/piper/zip/refs/heads/master [following]
--2026-06-22 06:26:22--  https://codeload.github.com/rhasspy/piper/zip/refs/heads/master
Resolving codeload.github.com (codeload.github.com)... 140.82.114.10
Connecting to codeload.github.com (codeload.github.com)|140.82.114.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘/content/piper.zip’

/content/piper.zip      [     <=>            ]  24.49M  16.3MB/s    in 1.5s    

2026-06-22 06:26:23 (16.3 MB/s) - ‘/content/piper.zip’ saved [25676019]

/content/piper
-- The C compiler identification is GNU 11.4.0
-- The CXX com

In [ ]:
import os
from google.colab import drive

print('--- 0. PREPARING ENVIRONMENT & PYTHON 3.10 ---')
!sudo apt-get update && apt-get install -y python3.10 python3.10-dev python3.10-distutils build-essential curl unzip espeak-ng espeak-ng-data
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!sudo update-alternatives --set python3 /usr/bin/python3.10
!python3.10 -m pip install "pip<24.1"

print('\n--- 1. RECOVERING GOOGLE DRIVE & DATA ---')
drive.mount('/content/drive', force_remount=True)
!unzip -q -o /content/drive/MyDrive/sepedi_tts_dataset.zip -d /content/

print('\n--- 2. DOWNLOADING PIPER (RECURSIVE CLONE) ---')
%cd /content
!rm -rf piper
!git clone --recursive https://github.com/rhasspy/piper.git

print('\n--- 3. INSTALLING DEPENDENCIES ---')
%cd /content/piper/src/python
!sed -i '/piper-phonemize/d' setup.py requirements.txt
!sed -i '/pytorch-lightning/d' setup.py requirements.txt
!sed -i '/torchmetrics/d' setup.py requirements.txt

!python3.10 -m pip install --no-cache-dir "numpy<2.0" "torch==1.13.1" "torchaudio==0.13.1" --index-url https://download.pytorch.org/whl/cu117
!python3.10 -m pip install --no-cache-dir pytorch-lightning==1.9.5 torchmetrics==0.11.4 lightning-utilities piper-phonemize-fix onnxruntime cython librosa scipy lazy_loader numba llvmlite joblib fsspec PyYAML tqdm

print('\n--- 4. BUILDING MONOTONIC ALIGNMENT ---')
# Build the extension inside the submodule
%cd /content/piper/src/python/piper_train/vits/monotonic_align
!python3.10 setup.py build_ext --inplace

# CRITICAL: Ensure the compiled .so file is in the right place for the piper_train package
!mkdir -p /content/piper/src/python/piper_train/vits/monotonic_align/monotonic_align
!cp core*.so /content/piper/src/python/piper_train/vits/monotonic_align/ || true
!cp core*.so /content/piper/src/python/piper_train/vits/monotonic_align/monotonic_align/ || true

%cd /content/piper/src/python
!python3.10 -m pip install -e . --no-deps

# Nuke the Lightning pkg_resources bug
!sed -i 's/__import__("pkg_resources").declare_namespace(__name__)/pass/g' /usr/local/lib/python3.10/dist-packages/lightning_fabric/__init__.py 2>/dev/null || true

print('\n--- 5. FIXING METADATA BUG ---')
with open('/content/sepedi_tts_dataset/metadata.csv', 'r', encoding='utf-8') as f:
    lines = [line.replace('.wav|', '|') for line in f.readlines()]
with open('/content/sepedi_tts_dataset/metadata.csv', 'w', encoding='utf-8') as f:
    f.writelines(lines)

print('\n--- 6. RUNNING PREPROCESSOR (SETSWANA PHONETICS) ---')
!rm -rf /content/training_ready
!export ESPEAK_DATA_PATH=/usr/lib/x86_64-linux-gnu/espeak-ng-data && python3.10 -m piper_train.preprocess \
  --language tn \
  --dataset-format ljspeech \
  --input-dir /content/sepedi_tts_dataset \
  --output-dir /content/training_ready \
  --single-speaker \
  --sample-rate 22050

print('\n--- 7. STARTING TRAINING ---')
!python3.10 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

--- 0. PREPARING ENVIRONMENT & PYTHON 3.10 ---
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,073 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.4 MB]
Fetched 13.5 MB in 3s (4,257 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease'

In [ ]:
from pathlib import Path
import os
import shutil
import textwrap

PIPER_ROOT = Path("/content/piper")
PY_ROOT = Path("/content/piper/src/python")
ALIGN_DIR = PY_ROOT / "piper_train" / "vits" / "monotonic_align"

print("=== 1. PREPARING ENVIRONMENT ===")
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10
!python3.10 -m pip install setuptools wheel cython "numpy<2.0" --quiet

ALIGN_DIR.mkdir(parents=True, exist_ok=True)

print("Generating core.pyx source directly...")
core_pyx_content = """
import numpy as np
cimport numpy as np

cdef void maximum_path_each(int[:, ::1] path, float[:, ::1] value, int t_y, int t_x) nogil:
    cdef int x, y
    cdef float max_val

    for y in range(t_y):
        path[y, 0] = 1

    for x in range(1, t_x):
        for y in range(x, t_y):
            max_val = value[y-1, x-1]
            if y > x:
                if value[y, x-1] > max_val:
                    max_val = value[y, x-1]
            value[y, x] += max_val

    x = t_x - 1
    y = t_y - 1
    path[y, x] = 1
    while x > 0:
        if y > x and value[y, x-1] < value[y-1, x-1]:
            y -= 1
        x -= 1
        path[y, x] = 1

def maximum_path_c(np.ndarray[int, ndim=3] paths, np.ndarray[float, ndim=3] values, np.ndarray[int, ndim=1] t_ys, np.ndarray[int, ndim=1] t_xs):
    cdef int b = paths.shape[0]
    cdef int i
    cdef int[:, ::1] path
    cdef float[:, ::1] value

    for i in range(b):
        path = paths[i]
        value = values[i]
        maximum_path_each(path, value, t_ys[i], t_xs[i])
"""
(ALIGN_DIR / "core.pyx").write_text(textwrap.dedent(core_pyx_content))

print("=== 2. COMPILING EXTENSION ===")
setup_code = """
from setuptools import setup, Extension
from Cython.Build import cythonize
import numpy
setup(
    ext_modules=cythonize([Extension('core', ['core.pyx'], include_dirs=[numpy.get_include()])], language_level=3)
)
"""
(ALIGN_DIR / "setup.py").write_text(textwrap.dedent(setup_code))

init_code = """
import torch
import numpy as np
try:
    from .core import maximum_path_c
except ImportError:
    import core as maximum_path_c

def maximum_path(neg_cent, mask):
    device, dtype = neg_cent.device, neg_cent.dtype
    neg_cent = neg_cent.data.cpu().numpy().astype('float32')
    path = np.zeros(mask.shape, dtype='int32')
    t_t_max = mask.sum(1)[:, 0].data.cpu().numpy().astype('int32')
    t_s_max = mask.sum(2)[:, 0].data.cpu().numpy().astype('int32')
    maximum_path_c(path, neg_cent, t_t_max, t_s_max)
    return torch.from_numpy(path).to(device=device, dtype=dtype)
"""
(ALIGN_DIR / "__init__.py").write_text(textwrap.dedent(init_code))

%cd "{ALIGN_DIR}"
!python3.10 setup.py build_ext --inplace

print("=== 3. FINAL INSTALL & TRAIN ===")
%cd "{PY_ROOT}"
!python3.10 -m pip install -e . --no-deps

check_cmd = "python3.10 -c 'import sys; sys.path.insert(0, \"/content/piper/src/python\"); from piper_train.vits.monotonic_align import maximum_path; print(\"SUCCESS\")'"
if os.system(check_cmd) == 0:
    print("Verification successful. Starting training...")
    !python3.10 -m piper_train \
      --dataset-dir /content/training_ready \
      --accelerator gpu \
      --devices 1 \
      --batch-size 16 \
      --validation-split 0.05 \
      --max_epochs 2000 \
      --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output
else:
    print("Import failed. Listing directory for debug:")
    !ls -R {ALIGN_DIR}

=== 1. PREPARING ENVIRONMENT ===
  Using cached pip-26.1.2-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.1.2
    Uninstalling pip-26.1.2:
      Successfully uninstalled pip-26.1.2
Generating core.pyx source directly...
=== 2. COMPILING EXTENSION ===
/content/piper/src/python/piper_train/vits/monotonic_align
Compiling core.pyx because it changed.
[1/1] Cythonizing core.pyx
performance hint: core.pyx:5:0: Exception check on 'maximum_path_each' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_each' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'maximum_path_each' to allow an error code to be returned.
performance hint: core.pyx:10:12: Use boundscheck(False) for faster access
performance hint: core.pyx:14:27: Use boundscheck(False) for faster access
p

In [ ]:
import torch
import numpy as np
from pathlib import Path
import textwrap
import os
from google.colab import drive

PY_ROOT = Path("/content/piper/src/python")
ALIGN_DIR = PY_ROOT / "piper_train/vits/monotonic_align"
INIT_FILE = ALIGN_DIR / "__init__.py"

print("=== 1. PREPARING DATASET ===")
if not os.path.exists('/content/sepedi_tts_dataset'):
    try:
        drive.mount('/content/drive')
        !unzip -q -o /content/drive/MyDrive/sepedi_tts_dataset.zip -d /content/
        print("Dataset unzipped successfully.")
    except Exception as e:
        print(f"Drive mount failed: {e}.")
else:
    print("Dataset already exists.")

print("=== 2. FORCING GPU PYTORCH INSTALLATION ===")
# Force uninstall existing CPU torch to prevent version hijacking
!python3.10 -m pip uninstall -y torch torchvision torchaudio
# Install specific CUDA 12.1 compatible build for Colab's hardware
!python3.10 -m pip install torch==2.1.0+cu121 torchaudio==2.1.0+cu121 --index-url https://download.pytorch.org/whl/cu121
!python3.10 -m pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 lightning-utilities --no-deps

print("\nDiagnostic Check:")
!python3.10 -c "import torch; print(f'Torch Version: {torch.__version__}'); print(f'CUDA Available: {torch.cuda.is_available()}'); print(f'Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')"

print("\n=== 3. PATCHING VITS ALIGNMENT ===")
ALIGN_DIR.mkdir(parents=True, exist_ok=True)
fallback_code = r'''
import torch
import numpy as np
def _maximum_path_numpy(value, mask):
    value = value * mask
    bsz, t_text, t_audio = value.shape
    path = np.zeros_like(value, dtype=np.float32)
    for b in range(bsz):
        valid_text = int(mask[b].sum(axis=1).clip(0, 1).sum())
        valid_audio = int(mask[b].sum(axis=0).clip(0, 1).sum())
        if valid_text <= 0 or valid_audio <= 0: continue
        v = value[b, :valid_text, :valid_audio]
        dp = np.full((valid_text, valid_audio), -1e9, dtype=np.float32)
        back = np.zeros((valid_text, valid_audio), dtype=np.int32)
        dp[0, 0] = v[0, 0]
        for j in range(1, valid_audio):
            dp[0, j] = dp[0, j - 1] + v[0, j]
            back[0, j] = 0
        for i in range(1, valid_text):
            for j in range(i, valid_audio):
                stay = dp[i, j - 1] if j - 1 >= 0 else -1e9
                move = dp[i - 1, j - 1] if j - 1 >= 0 else -1e9
                if move > stay: dp[i, j] = move + v[i, j]; back[i, j] = 1
                else: dp[i, j] = stay + v[i, j]; back[i, j] = 0
        i, j = valid_text - 1, valid_audio - 1
        while j >= 0:
            path[b, i, j] = 1.0
            if i > 0 and back[i, j] == 1: i -= 1
            j -= 1
    return path
def maximum_path(neg_cent, mask):
    device, dtype = neg_cent.device, neg_cent.dtype
    val_np = neg_cent.detach().cpu().numpy().astype(np.float32)
    mask_np = mask.detach().cpu().numpy().astype(np.float32)
    path = _maximum_path_numpy(val_np, mask_np)
    return torch.from_numpy(path).to(device=device, dtype=dtype)
'''
INIT_FILE.write_text(textwrap.dedent(fallback_code).strip() + "\n")

print("\n=== 4. STARTING TRAINING ===")
os.environ['PYTHONPATH'] = str(PY_ROOT)
!python3.10 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 8 \
  --validation-split 0.05 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

=== 1. PREPARING DATASET ===
Dataset already exists.
=== 2. FORCING GPU PYTORCH INSTALLATION ===
Found existing installation: torch 2.1.0+cu121
Uninstalling torch-2.1.0+cu121:
  Successfully uninstalled torch-2.1.0+cu121
Found existing installation: torchaudio 2.1.0+cu121
Uninstalling torchaudio-2.1.0+cu121:
  Successfully uninstalled torchaudio-2.1.0+cu121
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached torch-2.1.0%2Bcu121-cp310-cp310-linux_x86_64.whl (2200.6 MB)
  Using cached torchaudio-2.1.0%2Bcu121-cp310-cp310-linux_x86_64.whl (3.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchaudio]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
piper-train 1.0.0 requires piper-phonemize~=1.1.0, which is not installed.
piper-train 1.0.0 requires cython<1,>=0.29.0, but you have cython 3.2.5 which is incompatible.
piper-train 1.0.0 req

In [ ]:
print("=== GPU Hardware Check ===")
!nvidia-smi || echo "CRITICAL: No GPU hardware found. Please ensure a GPU runtime is selected (Runtime -> Change runtime type)."

print("\n=== Python 3.10 CUDA Check ===")
!python3.10 -c "import torch; print(f'Torch Version: {torch.__version__}'); print(f'CUDA Available: {torch.cuda.is_available()}'); print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"NONE\"}')"

=== GPU Hardware Check ===
/bin/bash: line 1: nvidia-smi: command not found
CRITICAL: No GPU hardware found. Please ensure a GPU runtime is selected (Runtime -> Change runtime type).

=== Python 3.10 CUDA Check ===
Torch Version: 2.0.1+cu118
CUDA Available: False
Device: NONE


### GPU and Torch Verification
This cell checks if the Colab instance has a GPU attached and if Python 3.10 can see it.

In [ ]:
print("=== 1. COLAB GPU CHECK ===")
!nvidia-smi || echo "No GPU detected. Please go to Runtime -> Change runtime type and select GPU."

print("\n=== 2. PYTHON 3.10 TORCH CHECK ===")
!python3.10 - <<'PY'
import torch
print(f"Torch Version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Torch CUDA Build: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
PY

=== 1. COLAB GPU CHECK ===
/bin/bash: line 1: nvidia-smi: command not found
No GPU detected. Please go to Runtime -> Change runtime type and select GPU.

=== 2. PYTHON 3.10 TORCH CHECK ===
/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
Torch Version: 2.11.0+cpu
CUDA available: False
Torch CUDA Build: None


NameError: name 'PY' is not defined

### Force Reinstall CUDA-Enabled PyTorch
If the check above shows `CUDA available: False`, run this cell to install the correct binaries.

In [ ]:
print('=== 1. INSTALLING PIP FOR PYTHON 3.10 ===')
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

print('\n=== 2. INSTALLING CUDA-ENABLED TORCH FOR PYTHON 3.10 ===')
# Using cu118 for compatibility with the specific piper-train requirements
!python3.10 -m pip install --upgrade pip setuptools wheel
!python3.10 -m pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 --index-url https://download.pytorch.org/whl/cu118

print('\n=== 3. INSTALLING PIPER TRAINING DEPENDENCIES ===')
!python3.10 -m pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 lightning-utilities piper-phonemize-fix onnxruntime cython librosa fsspec PyYAML tqdm --no-deps

print('\n=== 4. FINAL GPU VERIFICATION ===')
!python3.10 -c "import torch; print(f'Torch Version: {torch.__version__}'); print(f'CUDA Available: {torch.cuda.is_available()}'); print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"NONE\"}')"

=== 1. INSTALLING PIP FOR PYTHON 3.10 ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [wheel]

=== 2. INSTALLING CUDA-ENABLED TORCH FOR PYTHON 3.10 ===
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 4.9 MB/s  0:00:57
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 22.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 118.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.3/63.3 MB 60.7 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 140.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 20.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Resume Training
Once CUDA is verified as `True`, start the training process.

### Fixing `ModuleNotFoundError: No module named 'monotonic_align'`

In [ ]:
import os
from pathlib import Path
import shutil

PY_ROOT = Path('/content/piper/src/python')
TRAINING_READY = Path('/content/training_ready')
DRIVE_OUTPUT = '/content/drive/MyDrive/Sepedi_Voice_Output'

print('=== 1. REMOVING CONFLICTING MONOTONIC_ALIGN FOLDERS ===')
# Remove any internal/submodule monotonic_align to prevent conflicts
mono_align_internal_path = PY_ROOT / 'piper_train' / 'vits' / 'monotonic_align'
if mono_align_internal_path.exists():
    shutil.rmtree(mono_align_internal_path)
    print(f'Removed internal {mono_align_internal_path}')

print('\n=== 2. INSTALLING MONOTONIC-ALIGN AS A STANDALONE PACKAGE ===')
# Install the package directly from GitHub to ensure it's a proper Python package
# Use --force-reinstall to ensure a clean installation if previous attempts failed
!python3.10 -m pip install git+https://github.com/rhasspy/monotonic-align.git@vits-lightning-2 --force-reinstall

print('\n=== 3. PATCHING `models.py` FOR CORRECT IMPORT ===')
models_file = PY_ROOT / 'piper_train' / 'vits' / 'models.py'
if models_file.exists():
    # Read the content of models.py
    with open(models_file, 'r') as f:
        content = f.read()

    # Replace the relative import with a global import
    # This assumes 'from . import attentions, commons, modules, monotonic_align' is the line
    new_content = content.replace(
        'from . import attentions, commons, modules, monotonic_align',
        'from . import attentions, commons, modules\nimport monotonic_align'
    )

    # Write the modified content back to the file
    with open(models_file, 'w') as f:
        f.write(new_content)
    print(f'Patched {models_file} for global monotonic_align import.')
else:
    print(f'Error: {models_file} not found. Cannot patch import.')


print('\n=== 4. VERIFYING MONOTONIC_ALIGN IMPORT ===')
# Temporarily add PY_ROOT to PYTHONPATH for this check
os.environ['PYTHONPATH'] = str(PY_ROOT)
!python3.10 -c "import monotonic_align; print('Success: monotonic_align imported correctly!');"


print('\n=== 5. RE-INSTALLING PIPER_TRAIN IN EDITABLE MODE ===')
%cd {PY_ROOT}
!python3.10 -m pip install -e . --no-deps


print('\n=== 6. STARTING TRAINING (50-STEP SMOKE TEST) ===')
# Ensure the preprocessed data is ready
if not TRAINING_READY.exists():
    print(f'Warning: {TRAINING_READY} not found. Running preprocess step...')
    !PYTHONPATH={PY_ROOT} python3.10 -m piper_train.preprocess \
      --language tn \
      --dataset-format ljspeech \
      --input-dir /content/sepedi_tts_dataset \
      --output-dir {TRAINING_READY} \
      --single-speaker \
      --sample-rate 22050

# Launch training
!python3.10 -m piper_train \
  --dataset-dir {TRAINING_READY} \
  --accelerator gpu \
  --devices 1 \
  --batch-size 8 \
  --max_steps 50 \
  --default_root_dir {DRIVE_OUTPUT}

In [ ]:
import os
from pathlib import Path

# 1. Setup paths
PY_ROOT = Path("/content/piper/src/python")
MONO_SRC = PY_ROOT / "piper_train" / "vits" / "monotonic_align"
os.environ['PYTHONPATH'] = str(PY_ROOT)

print("=== BUILDING ALIGNMENT TOOL FROM SOURCE ===")
if MONO_SRC.exists():
    %cd {MONO_SRC}
    # Clean and build
    !rm -rf build/ core*.so
    !python3.10 setup.py build_ext --inplace
    # Copy to both local and parent for safety
    !cp core*.so ../
    !cp core*.so {PY_ROOT}/
else:
    print(f"Error: {MONO_SRC} not found!")

# 2. Fix the import in models.py to handle the local module correctly
MODELS_PY = PY_ROOT / "piper_train" / "vits" / "models.py"
if MODELS_PY.exists():
    # Ensure we use an absolute-style import for the built-in module
    !sed -i "s/from . import attentions, commons, modules, monotonic_align/from . import attentions, commons, modules\nfrom . import monotonic_align/g" {MODELS_PY}

# 3. Final linking
%cd {PY_ROOT}
print("\n=== LINKING PIPER MODULE ===")
!python3.10 -m pip install -e . --no-deps

# 4. Verify and Start Training
print("\n=== VERIFYING IMPORT ===")
!python3.10 -c "import sys; sys.path.append('/content/piper/src/python'); from piper_train.vits import monotonic_align; print('Success: monotonic_align found')"

print("\n=== STARTING SEPEDI TTS TRAINING (GPU) ===")
!python3.10 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 8 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

=== BUILDING ALIGNMENT TOOL FROM SOURCE ===
Error: /content/piper/src/python/piper_train/vits/monotonic_align not found!
/content/piper/src/python

=== LINKING PIPER MODULE ===
Obtaining file:///content/piper/src/python
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for piper_train (pyproject.toml) ... done
  Created wheel for piper_train: filename=piper_train-1.0.0-0.editable-py3-none-any.whl size=3384 sha256=242ade51b98bb7bb3902816638ebbecd7cf5475c101c58a7adf4b7770ba6755f
  Stored in directory: /tmp/pip-ephem-wheel-cache-gzlu3tjb/wheels/c5/4f/48/55937acbb532aa3bc9506b36ab72ddc60918c2cd39869aed9c
Successfully built piper_train
  Attempting uninstall: piper_train
    Found existing installation: piper_train 1.0.0
    Uninstalling piper_train-1.0.0:
      Successfully uninstalled piper_train-1.0.0

=== VERIFY

In [ ]:
import os
import shutil
import textwrap
from pathlib import Path

PY_ROOT = Path("/content/piper/src/python")
# The correct nested path within the repository
SRC_ALIGN = PY_ROOT / "piper_train" / "vits" / "monotonic_align"
# The target top-level path that models.py expects
TOP_ALIGN = PY_ROOT / "monotonic_align"

print("=== 1. PREPARING TOP-LEVEL monotonic_align PACKAGE ===")
# Ensure the top-level directory exists
TOP_ALIGN.mkdir(parents=True, exist_ok=True)

# Find the compiled extension in the source folder or build it
so_files = list(SRC_ALIGN.rglob("core*.so"))
if not so_files:
    if SRC_ALIGN.exists():
        print(f"Extension not found in {SRC_ALIGN}. Attempting build...")
        %cd {SRC_ALIGN}
        !python3.10 setup.py build_ext --inplace
        so_files = list(SRC_ALIGN.rglob("core*.so"))
    else:
        print(f"Warning: Source directory {SRC_ALIGN} not found.")

if not so_files:
    print("Warning: Could not find or build core*.so. Training will use slow NumPy fallback.")
else:
    # Copy compiled extension to the top-level folder
    for so in so_files:
        shutil.copy2(so, TOP_ALIGN / so.name)
    print(f"Copied extension to {TOP_ALIGN}")

# 2. Create the __init__.py that Piper's models.py expects
init_code = r'''
try:
    from .core import maximum_path_c
except Exception:
    try:
        import core as maximum_path_c
    except Exception:
        maximum_path_c = None

import torch
import numpy as np

def _maximum_path_numpy(value, mask):
    value = value * mask
    bsz, t_text, t_audio = value.shape
    path = np.zeros_like(value, dtype=np.float32)
    for b in range(bsz):
        valid_text = int(mask[b].sum(axis=1).clip(0, 1).sum())
        valid_audio = int(mask[b].sum(axis=0).clip(0, 1).sum())
        if valid_text <= 0 or valid_audio <= 0: continue
        v = value[b, :valid_text, :valid_audio]
        dp = np.full((valid_text, valid_audio), -1e9, dtype=np.float32)
        back = np.zeros((valid_text, valid_audio), dtype=np.int32)
        dp[0, 0] = v[0, 0]
        for j in range(1, valid_audio): dp[0, j] = dp[0, j - 1] + v[0, j]
        for i in range(1, valid_text):
            for j in range(i, valid_audio):
                stay = dp[i, j - 1] if j - 1 >= 0 else -1e9
                move = dp[i - 1, j - 1] if j - 1 >= 0 else -1e9
                if move > stay: dp[i, j] = move + v[i, j]; back[i, j] = 1
                else: dp[i, j] = stay + v[i, j]; back[i, j] = 0
        i, j = valid_text - 1, valid_audio - 1
        while j >= 0:
            path[b, i, j] = 1.0
            if i > 0 and back[i, j] == 1: i -= 1
            j -= 1
    return path

def maximum_path(neg_cent, mask):
    device, dtype = neg_cent.device, neg_cent.dtype
    if maximum_path_c is not None:
        path = np.zeros(mask.shape, dtype='int32')
        neg_cent_np = neg_cent.data.cpu().numpy().astype('float32')
        t_t_max = mask.sum(1)[:, 0].data.cpu().numpy().astype('int32')
        t_s_max = mask.sum(2)[:, 0].data.cpu().numpy().astype('int32')
        maximum_path_c(path, neg_cent_np, t_t_max, t_s_max)
        return torch.from_numpy(path).to(device=device, dtype=dtype)
    return torch.from_numpy(_maximum_path_numpy(neg_cent.detach().cpu().numpy(), mask.detach().cpu().numpy())).to(device=device, dtype=dtype)
'''
(TOP_ALIGN / "__init__.py").write_text(textwrap.dedent(init_code))

print("\n=== 3. STARTING TRAINING WITH UPDATED PATHS ===")
%cd {PY_ROOT}
os.environ['PYTHONPATH'] = str(PY_ROOT)
!python3.10 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 8 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

=== 1. PREPARING TOP-LEVEL monotonic_align PACKAGE ===
Extension not found. Attempting a quick rebuild...
[Errno 2] No such file or directory: '/content/piper/src/python/piper_train/vits/monotonic_align'
/content/piper/src/python
/usr/local/lib/python3.10/dist-packages/setuptools/dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
!!

        ********************************************************************************
        Please consider removing the following classifiers in favor of a SPDX license expression:

        License :: OSI Approved :: MIT License

        See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
        ********************************************************************************

!!
  self._finalize_license_expression()
running build_ext


FileNotFoundError: [Errno 2] No such file or directory: '/content/piper/src/python/monotonic_align/__init__.py'

In [ ]:
from pathlib import Path
import textwrap
import os
import shutil

PY_ROOT = Path("/content/piper/src/python")
TOP_ALIGN = PY_ROOT / "monotonic_align"
DATASET_DIR = Path("/content/sepedi_tts_dataset")

print("=== 1. PREPARING DATASET AND DIRECTORIES ===")
if not DATASET_DIR.exists():
    from google.colab import drive
    drive.mount('/content/drive')
    !unzip -q -o /content/drive/MyDrive/sepedi_tts_dataset.zip -d /content/

print("\n=== 2. APPLYING NUCLEAR NUMERICAL STABILITY PATCH ===")
transforms_path = PY_ROOT / "piper_train" / "vits" / "transforms.py"
if transforms_path.exists():
    # Read file and replace unstable code with safe alternatives
    content = transforms_path.read_text()

    # 1. Handle NaNs in theta
    content = content.replace("theta_minus_one = theta - 1", "theta = torch.nan_to_num(theta); theta_minus_one = theta - 1")

    # 2. Replace the discriminant calculation and assertion with a safe clamped version
    # This targets: discriminant = torch.pow(theta_minus_one, 2) - 4 * ...
    # We'll use a broader replacement for the assertion itself
    content = content.replace("assert (discriminant >= 0).all()", "discriminant = torch.nan_to_num(discriminant, nan=1e-6).clamp(min=1e-6)")

    # 3. Handle sqrt(discriminant) if assertion was skipped or removed
    content = content.replace("torch.sqrt(discriminant)", "torch.sqrt(torch.clamp(discriminant, min=1e-6))")

    transforms_path.write_text(content)
    print("Nuclear stability patch applied to transforms.py")

print("\n=== 3. RE-CREATING MONOTONIC ALIGN FALLBACK ===")
TOP_ALIGN.mkdir(parents=True, exist_ok=True)
init_code = r'''
import torch
import numpy as np
def _maximum_path_numpy(value, mask):
    value = value * mask
    bsz, t_text, t_audio = value.shape
    path = np.zeros_like(value, dtype=np.float32)
    for b in range(bsz):
        valid_text = int(mask[b].sum(axis=1).clip(0, 1).sum())
        valid_audio = int(mask[b].sum(axis=0).clip(0, 1).sum())
        if valid_text <= 0 or valid_audio <= 0: continue
        v = value[b, :valid_text, :valid_audio]
        dp = np.full((valid_text, valid_audio), -1e9, dtype=np.float32)
        back = np.zeros((valid_text, valid_audio), dtype=np.int32)
        dp[0, 0] = v[0, 0]
        for j in range(1, valid_audio): dp[0, j] = dp[0, j - 1] + v[0, j]
        for i in range(1, valid_text):
            for j in range(i, valid_audio):
                stay = dp[i, j - 1] if j - 1 >= 0 else -1e9
                move = dp[i - 1, j - 1] if j - 1 >= 0 else -1e9
                if move > stay: dp[i, j] = move + v[i, j]; back[i, j] = 1
                else: dp[i, j] = stay + v[i, j]; back[i, j] = 0
        i, j = valid_text - 1, valid_audio - 1
        while j >= 0:
            path[b, i, j] = 1.0
            if i > 0 and back[i, j] == 1: i -= 1
            j -= 1
    return path
def maximum_path(neg_cent, mask):
    device, dtype = neg_cent.device, neg_cent.dtype
    value_np = neg_cent.detach().cpu().numpy().astype(np.float32)
    mask_np = mask.detach().cpu().numpy().astype(np.float32)
    path_np = _maximum_path_numpy(value_np, mask_np)
    return torch.from_numpy(path_np).to(device=device, dtype=dtype)
'''
(TOP_ALIGN / "__init__.py").write_text(textwrap.dedent(init_code).strip() + "\n")

print("\n=== 4. STARTING STABILIZED TRAINING ===")
# Running with validation-split 0.0 and smaller LR to prevent early collapse
!PYTHONPATH=/content/piper/src/python:$PYTHONPATH python3.10 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 1 \
  --validation-split 0.0 \
  --num-test-examples 0 \
  --num_sanity_val_steps 0 \
  --gradient_clip_val 0.1 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

=== 1. PREPARING DATASET AND DIRECTORIES ===

=== 2. APPLYING NUCLEAR NUMERICAL STABILITY PATCH ===
Nuclear stability patch applied to transforms.py

=== 3. RE-CREATING MONOTONIC ALIGN FALLBACK ===

=== 4. STARTING STABILIZED TRAINING ===
DEBUG:piper_train:Namespace(dataset_dir='/content/training_ready', checkpoint_epochs=None, quality='medium', resume_from_single_speaker_checkpoint=None, logger=True, enable_checkpointing=True, default_root_dir='/content/drive/MyDrive/Sepedi_Voice_Output', gradient_clip_val=0.1, gradient_clip_algorithm=None, num_nodes=1, num_processes=None, devices='1', gpus=None, auto_select_gpus=None, tpu_cores=None, ipus=None, enable_progress_bar=True, overfit_batches=0.0, track_grad_norm=-1, check_val_every_n_epoch=1, fast_dev_run=False, accumulate_grad_batches=None, max_epochs=2000, min_epochs=None, max_steps=-1, min_steps=None, max_time=None, limit_train_batches=None, limit_val_batches=None, limit_test_batches=None, limit_predict_batches=None, val_check_interval=

In [ ]:
import os
import json
import librosa
import numpy as np
from tqdm import tqdm
from pathlib import Path

# Configuration
DATASET_JSONL = "/content/training_ready/dataset.jsonl"
CLEAN_JSONL = "/content/training_ready/dataset_cleaned.jsonl"
MIN_DURATION = 0.35  # seconds
MAX_DURATION = 15.0  # seconds
MAX_PHONEME_LEN = 350

print("=== DATASET INTEGRITY AUDIT ===")
problematic_files = []
cleaned_data = []

if not os.path.exists(DATASET_JSONL):
    print(f"Error: {DATASET_JSONL} not found. Please run preprocessor first.")
else:
    with open(DATASET_JSONL, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    print(f"Scanning {len(lines)} samples...")
    for line in tqdm(lines):
        data = json.loads(line)
        wav_path = data.get('audio_path')
        text = data.get('text', "")
        phoneme_ids = data.get('phoneme_ids', [])

        try:
            # 1. Check text and phoneme integrity
            if len(text.strip()) == 0:
                problematic_files.append((wav_path, "Empty text field"))
                continue
            if len(phoneme_ids) == 0:
                problematic_files.append((wav_path, "Empty phoneme_ids"))
                continue
            if len(phoneme_ids) > MAX_PHONEME_LEN:
                problematic_files.append((wav_path, f"Phoneme sequence too long: {len(phoneme_ids)}"))
                continue

            # 2. Check if file exists
            if not os.path.exists(wav_path):
                problematic_files.append((wav_path, "Missing file"))
                continue

            # 3. Check for corruption/validity
            y, sr = librosa.load(wav_path, sr=None)
            duration = librosa.get_duration(y=y, sr=sr)

            # 4. Check for silence or NaNs in audio
            if np.isnan(y).any():
                problematic_files.append((wav_path, "NaNs in audio data"))
                continue

            peak = np.max(np.abs(y))
            if peak < 0.01:
                problematic_files.append((wav_path, "Silent audio"))
                continue
            if peak > 1.0:
                # VITS is sensitive to clipping/over-normalized audio early on
                problematic_files.append((wav_path, f"Audio peak > 1.0 ({peak:.4f})"))
                continue

            # 5. Check duration constraints
            if duration < MIN_DURATION or duration > MAX_DURATION:
                problematic_files.append((wav_path, f"Invalid duration: {duration:.2f}s"))
                continue

            cleaned_data.append(line)

        except Exception as e:
            problematic_files.append((wav_path, f"Error: {str(e)}"))

    print(f"\nAudit Complete!")
    print(f"Total samples scanned: {len(lines)}")
    print(f"Problematic samples removed: {len(problematic_files)}")

    if problematic_files:
        print("\nFirst 10 problematic samples:")
        for path, reason in problematic_files[:10]:
            print(f" - {path}: {reason}")

    with open(CLEAN_JSONL, 'w', encoding='utf-8') as f:
        for line in cleaned_data:
            f.write(line)

    # Replace the original with the cleaned version
    os.replace(CLEAN_JSONL, DATASET_JSONL)
    print(f"\nCleaned dataset saved back to {DATASET_JSONL}")

=== DATASET INTEGRITY AUDIT ===
Scanning 663 samples...


100%|██████████| 663/663 [00:13<00:00, 48.02it/s]


Audit Complete!
Total samples scanned: 663
Problematic samples removed: 0

Cleaned dataset saved back to /content/training_ready/dataset.jsonl


In [ ]:
from pathlib import Path
import os
import shutil
import textwrap
from google.colab import drive

PIPER_BASE = Path("/content/piper")
PY_ROOT = PIPER_BASE / "src" / "python"
DATASET_SRC = Path("/content/sepedi_tts_dataset")
TRAINING_READY = Path("/content/training_ready")
TOP_ALIGN = PY_ROOT / "monotonic_align"

# Ensure CUDA drivers are in the path for the custom kernel
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')

print("=== 0. ENSURE PIPER REPOSITORY EXISTS ===")
if not PIPER_BASE.exists():
    %cd /content
    !git clone https://github.com/rhasspy/piper.git

print("=== 1. VERIFY DATASET AND DRIVE ===")
if not DATASET_SRC.exists():
    drive.mount("/content/drive", force_remount=True)
    !unzip -q -o /content/drive/MyDrive/sepedi_tts_dataset.zip -d /content/

print("\n=== 2. CONFIGURE PYTHON 3.10 ENVIRONMENT ===")
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10
!python3.10 -m pip install "pip<24.1" "numpy<2.0" --force-reinstall

# Re-installing torch with explicit index to ensure CUDA support
!python3.10 -m pip install torch==2.1.0+cu121 torchaudio==2.1.0+cu121 --index-url https://download.pytorch.org/whl/cu121
!python3.10 -m pip install msgpack audioread pooch decorator numba==0.58.1 llvmlite
!python3.10 -m pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 piper-phonemize-fix onnxruntime cython librosa==0.9.2 scipy fsspec PyYAML tqdm lazy_loader joblib --no-deps
!python3.10 -m pip install soundfile lightning-utilities --quiet

print("\n=== 3. CREATE MONOTONIC ALIGN FALLBACK ===")
TOP_ALIGN.mkdir(parents=True, exist_ok=True)
init_code = r'''
import torch
import numpy as np
def _maximum_path_numpy(value, mask):
    value = value * mask
    bsz, t_text, t_audio = value.shape
    path = np.zeros_like(value, dtype=np.float32)
    for b in range(bsz):
        valid_text = int(mask[b].sum(axis=1).clip(0, 1).sum())
        valid_audio = int(mask[b].sum(axis=0).clip(0, 1).sum())
        if valid_text <= 0 or valid_audio <= 0: continue
        v = value[b, :valid_text, :valid_audio]
        dp = np.full((valid_text, valid_audio), -1e9, dtype=np.float32)
        back = np.zeros((valid_text, valid_audio), dtype=np.int32)
        dp[0, 0] = v[0, 0]
        for j in range(1, valid_audio): dp[0, j] = dp[0, j - 1] + v[0, j]
        for i in range(1, valid_text):
            for j in range(i, valid_audio):
                stay = dp[i, j - 1] if j - 1 >= 0 else -1e9
                move = dp[i - 1, j - 1] if j - 1 >= 0 else -1e9
                if move > stay: dp[i, j] = move + v[i, j]; back[i, j] = 1
                else: dp[i, j] = stay + v[i, j]; back[i, j] = 0
        i, j = valid_text - 1, valid_audio - 1
        while j >= 0:
            path[b, i, j] = 1.0
            if i > 0 and back[i, j] == 1: i -= 1
            j -= 1
    return path
def maximum_path(neg_cent, mask):
    device, dtype = neg_cent.device, neg_cent.dtype
    val_np = neg_cent.detach().cpu().numpy().astype(np.float32)
    mask_np = mask.detach().cpu().numpy().astype(np.float32)
    path = _maximum_path_numpy(val_np, mask_np)
    return torch.from_numpy(path).to(device=device, dtype=dtype)
'''
(TOP_ALIGN / "__init__.py").write_text(textwrap.dedent(init_code).strip())

print("\n=== 4. PATCH MODELS AND REINSTALL ==")
models_path = PY_ROOT / "piper_train/vits/models.py"
if models_path.exists():
    m_text = models_path.read_text()
    if "import monotonic_align" not in m_text:
        m_text = m_text.replace("from . import attentions, commons, modules, monotonic_align", "from . import attentions, commons, modules\nimport monotonic_align")
        models_path.write_text(m_text)

%cd "{PY_ROOT}"
!python3.10 -m pip install -e . --no-deps

print("\n=== 5. GPU VERIFICATION & SMOKE TEST ===")
!python3.10 -c "import torch; print(f'GPU Available: {torch.cuda.is_available()}'); exit(0 if torch.cuda.is_available() else 1)"

if TRAINING_READY.exists(): shutil.rmtree(TRAINING_READY)
!PYTHONPATH={PY_ROOT} python3.10 -m piper_train.preprocess --language tn --dataset-format ljspeech --input-dir {DATASET_SRC} --output-dir {TRAINING_READY} --single-speaker --sample-rate 22050
!PYTHONPATH={PY_ROOT} python3.10 -m piper_train --dataset-dir {TRAINING_READY} --accelerator gpu --devices 1 --batch-size 1 --max_steps 50 --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

=== 0. ENSURE PIPER REPOSITORY EXISTS ===
=== 1. VERIFY DATASET AND DRIVE ===

=== 2. CONFIGURE PYTHON 3.10 ENVIRONMENT ===
  Using cached pip-26.1.2-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
  Using cached pip-24.0-py3-none-any.whl.metadata (3.6 kB)
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached pip-24.0-py3-none-any.whl (2.1 MB)
Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.1.2
    Uninstalling pip-26.1.2:
      Successfully uninstalled pip-26.1.2
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
from pathlib import Path

print('=== 1. INSTALLING CUDA-ENABLED TORCH ===')
# Re-verify and install Torch 2.0.1 with CUDA 11.8 support
!python3.10 -m pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 --index-url https://download.pytorch.org/whl/cu118

# Re-install core dependencies ensuring they don't override Torch
!python3.10 -m pip install pytorch-lightning==1.9.5 torchmetrics==0.11.4 lightning-utilities piper-phonemize-fix onnxruntime cython librosa==0.9.2 fsspec PyYAML tqdm --no-deps

print('\n=== 2. VERIFYING HARDWARE ACCELERATION ===')
# Check if torch can actually see the GPU
!python3.10 -c "import torch; print(f'Torch Version: {torch.__version__}'); print(f'CUDA Available: {torch.cuda.is_available()}'); print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"NONE\"}')"

print('\n=== 3. RESTORING MONOTONIC ALIGN FALLBACK ===')
PY_ROOT = Path('/content/piper/src/python')
TOP_ALIGN = PY_ROOT / 'monotonic_align'
TOP_ALIGN.mkdir(parents=True, exist_ok=True)

init_code = r'''
import torch
import numpy as np
def _maximum_path_numpy(value, mask):
    value = value * mask
    bsz, t_text, t_audio = value.shape
    path = np.zeros_like(value, dtype=np.float32)
    for b in range(bsz):
        valid_text = int(mask[b].sum(axis=1).clip(0, 1).sum())
        valid_audio = int(mask[b].sum(axis=0).clip(0, 1).sum())
        if valid_text <= 0 or valid_audio <= 0: continue
        v = value[b, :valid_text, :valid_audio]
        dp = np.full((valid_text, valid_audio), -1e9, dtype=np.float32)
        back = np.zeros((valid_text, valid_audio), dtype=np.int32)
        dp[0, 0] = v[0, 0]
        for j in range(1, valid_audio): dp[0, j] = dp[0, j - 1] + v[0, j]
        for i in range(1, valid_text):
            for j in range(i, valid_audio):
                stay = dp[i, j - 1] if j - 1 >= 0 else -1e9
                move = dp[i - 1, j - 1] if j - 1 >= 0 else -1e9
                if move > stay: dp[i, j] = move + v[i, j]; back[i, j] = 1
                else: dp[i, j] = stay + v[i, j]; back[i, j] = 0
        i, j = valid_text - 1, valid_audio - 1
        while j >= 0:
            path[b, i, j] = 1.0
            if i > 0 and back[i, j] == 1: i -= 1
            j -= 1
    return path
def maximum_path(neg_cent, mask):
    device, dtype = neg_cent.device, neg_cent.dtype
    val_np = neg_cent.detach().cpu().numpy().astype(np.float32)
    mask_np = mask.detach().cpu().numpy().astype(np.float32)
    path = _maximum_path_numpy(val_np, mask_np)
    return torch.from_numpy(path).to(device=device, dtype=dtype)
'''
(TOP_ALIGN / '__init__.py').write_text(init_code)

print('\n=== 4. LAUNCHING 50-STEP SMOKE TEST ===')
os.environ['PYTHONPATH'] = str(PY_ROOT)
!python3.10 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 1 \
  --max_steps 50 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

=== 1. ENSURING PIP AND REINSTALLING CUDA-ENABLED TORCH ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [wheel]
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB ?  0:01:23
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 52.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.3/63.3 MB 54.4 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 61.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 41.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 109.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 18.2 MB/s  0:00:00
  Created wheel

In [ ]:
import os
from pathlib import Path

PIPER_ROOT = Path("/content/piper/src/python")
TRAINING_DIR = Path("/content/training_ready")
DRIVE_OUTPUT = "/content/drive/MyDrive/Sepedi_Voice_Output"

# Ensure environment variables are set correctly for the training script
os.environ['PYTHONPATH'] = str(PIPER_ROOT)

print("=== 4. LAUNCHING 50-STEP SMOKE TEST ===")
# Preprocess (Setswana Proxy)
!python3.10 -m piper_train.preprocess \
  --language tn \
  --dataset-format ljspeech \
  --input-dir /content/sepedi_tts_dataset \
  --output-dir {TRAINING_DIR} \
  --single-speaker \
  --sample-rate 22050

# Training Execution
!python3.10 -m piper_train \
  --dataset-dir {TRAINING_DIR} \
  --accelerator gpu \
  --devices 1 \
  --batch-size 1 \
  --max_steps 50 \
  --default_root_dir {DRIVE_OUTPUT}

In [ ]:
from pathlib import Path
import pandas as pd
import os

# Define the root directory for outputs
OUTPUT_ROOT_DIR = Path('/content/drive/MyDrive/Sepedi_Voice_Output')
LOGS_DIR = OUTPUT_ROOT_DIR / 'lightning_logs'

print(f"Checking logs in: {LOGS_DIR}")

if LOGS_DIR.exists():
    # Find the most recent version directory
    version_dirs = sorted([d for d in LOGS_DIR.iterdir() if d.is_dir() and d.name.startswith('version_')])

    if version_dirs:
        latest_version_dir = version_dirs[-1]
        metrics_path = latest_version_dir / 'metrics.csv'

        print(f"Found latest version directory: {latest_version_dir.name}")

        if metrics_path.exists():
            print(f"Loading metrics from: {metrics_path}")
            metrics_df = pd.read_csv(metrics_path)

            # Display the last few rows to show recent progress
            print("\nMost recent training metrics:")
            display(metrics_df.tail())

            # Optionally, display a summary of key metrics over time
            print("\nSummary of training progress:")
            if 'train_loss_step' in metrics_df.columns:
                display(metrics_df.dropna(subset=['train_loss_step']).set_index('epoch')['train_loss_step'].plot(title='Training Loss Over Epochs'))
            elif 'train_loss_epoch' in metrics_df.columns:
                display(metrics_df.dropna(subset=['train_loss_epoch']).set_index('epoch')['train_loss_epoch'].plot(title='Training Loss Over Epochs'))

            if 'val_loss' in metrics_df.columns:
                display(metrics_df.dropna(subset=['val_loss']).set_index('epoch')['val_loss'].plot(title='Validation Loss Over Epochs'))

        else:
            print(f"Metrics file not found in {latest_version_dir}")
    else:
        print("No version directories found in lightning_logs.")
else:
    print(f"Error: Log directory {LOGS_DIR} does not exist. Training might not have started or completed correctly.")

### GPU Verification

Let's check if the GPU is now available after potentially changing the runtime type.

In [ ]:
import os
import textwrap
from pathlib import Path

# 1. Define Paths
PY_ROOT = Path('/content/piper/src/python')
TOP_ALIGN = PY_ROOT / 'monotonic_align'

print('=== 1. PREPARING MONOTONIC ALIGN FALLBACK ===')
# Ensure the directory exists so the import doesn't fail
TOP_ALIGN.mkdir(parents=True, exist_ok=True)

# Create the __init__.py with a robust NumPy fallback to bypass compilation issues
init_code = r'''
import torch
import numpy as np

def _maximum_path_numpy(value, mask):
    value = value * mask
    bsz, t_text, t_audio = value.shape
    path = np.zeros_like(value, dtype=np.float32)
    for b in range(bsz):
        valid_text = int(mask[b].sum(axis=1).clip(0, 1).sum())
        valid_audio = int(mask[b].sum(axis=0).clip(0, 1).sum())
        if valid_text <= 0 or valid_audio <= 0: continue
        v = value[b, :valid_text, :valid_audio]
        dp = np.full((valid_text, valid_audio), -1e9, dtype=np.float32)
        back = np.zeros((valid_text, valid_audio), dtype=np.int32)
        dp[0, 0] = v[0, 0]
        for j in range(1, valid_audio): dp[0, j] = dp[0, j - 1] + v[0, j]
        for i in range(1, valid_text):
            for j in range(i, valid_audio):
                stay = dp[i, j - 1]
                move = dp[i - 1, j - 1]
                if move > stay:
                    dp[i, j] = move + v[i, j]
                    back[i, j] = 1
                else:
                    dp[i, j] = stay + v[i, j]
                    back[i, j] = 0
        i, j = valid_text - 1, valid_audio - 1
        while j >= 0:
            path[b, i, j] = 1.0
            if i > 0 and back[i, j] == 1: i -= 1
            j -= 1
    return path

def maximum_path(neg_cent, mask):
    device, dtype = neg_cent.device, neg_cent.dtype
    val_np = neg_cent.detach().cpu().numpy().astype(np.float32)
    mask_np = mask.detach().cpu().numpy().astype(np.float32)
    path = _maximum_path_numpy(val_np, mask_np)
    return torch.from_numpy(path).to(device=device, dtype=dtype)
'''
(TOP_ALIGN / '__init__.py').write_text(textwrap.dedent(init_code))
print(f'Fallback created at {TOP_ALIGN}')

print('\n=== 2. PATCHING MODELS.PY ===')
MODELS_PY = PY_ROOT / 'piper_train' / 'vits' / 'models.py'
if MODELS_PY.exists():
    # Change the relative import to a top-level import for our fallback
    !sed -i 's/from . import attentions, commons, modules, monotonic_align/from . import attentions, commons, modules\nimport monotonic_align/g' {MODELS_PY}
    print('Import patch applied.')

print('\n=== 3. STARTING TRAINING ===')
os.environ['PYTHONPATH'] = f"{PY_ROOT}:{os.environ.get('PYTHONPATH', '')}"

!python3.10 -m piper_train \
  --dataset-dir /content/training_ready \
  --accelerator gpu \
  --devices 1 \
  --batch-size 8 \
  --validation-split 0.05 \
  --num-test-examples 5 \
  --max_epochs 2000 \
  --default_root_dir /content/drive/MyDrive/Sepedi_Voice_Output

# New Section